# Plateau Decay Amount and Metrics Analysis

This notebook analyzes the plateau-region decay amount sweep.

Design:

- Stable training continues to step 1400.
- At several checkpoints, probes branch from the current model state.
- Each probe trains for 150 steps with a different final LR ratio.
- `final_lr_ratio = 1.0` is the stable-continuation baseline for that same checkpoint.

The key target is not raw final validation loss across checkpoints, because later checkpoints are expected to be better. The key target is:

```text
decay_advantage = val_after_decay_ratio - val_after_ratio_1.0
```

Negative values mean the decay ratio beats stable continuation from the same checkpoint.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

In [ ]:
# Set RUN_NAME after the run finishes, or leave as None to use the latest matching run.
RUN_NAME = None
LOG_ROOT = Path("../logs")
PREFIX = "wsd_intermediate_plateau_amount_sweep"

if RUN_NAME is None:
    candidates = sorted(p for p in LOG_ROOT.iterdir() if p.is_dir() and p.name.startswith(PREFIX) and (p / "decay_amount_sweep.jsonl").exists())
    if not candidates:
        raise FileNotFoundError(f"No completed {PREFIX} runs found under {LOG_ROOT}")
    LOG_DIR = candidates[-1]
else:
    LOG_DIR = LOG_ROOT / RUN_NAME

assert (LOG_DIR / "metrics.jsonl").exists(), "Missing metrics.jsonl"
assert (LOG_DIR / "decay_amount_sweep.jsonl").exists(), "Missing decay_amount_sweep.jsonl"
assert (LOG_DIR / "decay_amount_trajectory.jsonl").exists(), "Missing decay_amount_trajectory.jsonl"

def read_jsonl(path):
    with open(path) as f:
        return pd.DataFrame(json.loads(line) for line in f if line.strip())

metrics = read_jsonl(LOG_DIR / "metrics.jsonl")
sweep = read_jsonl(LOG_DIR / "decay_amount_sweep.jsonl")
trajectory = read_jsonl(LOG_DIR / "decay_amount_trajectory.jsonl")

val_rows = metrics[metrics["validation_loss"].notna()][["step", "validation_loss"]].rename(
    columns={"step": "probe_start_step", "validation_loss": "probe_start_val_loss"}
)
sweep = sweep.merge(val_rows, on="probe_start_step", how="left")
trajectory = trajectory.merge(val_rows, on="probe_start_step", how="left")
trajectory["global_step_equivalent"] = trajectory["probe_start_step"] + trajectory["probe_decay_step"]

baseline = sweep[sweep["final_lr_ratio"] == 1.0][["probe_start_step", "probe_final_val_loss"]].rename(
    columns={"probe_final_val_loss": "stable_continuation_val_loss"}
)
sweep = sweep.merge(baseline, on="probe_start_step", how="left")
sweep["decay_advantage"] = sweep["probe_final_val_loss"] - sweep["stable_continuation_val_loss"]
sweep["decay_val_improvement"] = sweep["probe_start_val_loss"] - sweep["probe_final_val_loss"]
sweep["log_final_lr_ratio"] = np.log10(sweep["final_lr_ratio"])
sweep["lr_drop_fraction"] = 1 - sweep["final_lr_ratio"]

traj_base = trajectory[trajectory["final_lr_ratio"] == 1.0][["probe_start_step", "probe_decay_step", "probe_validation_loss"]].rename(
    columns={"probe_validation_loss": "stable_continuation_validation_loss"}
)
trajectory = trajectory.merge(traj_base, on=["probe_start_step", "probe_decay_step"], how="left")
trajectory["decay_advantage"] = trajectory["probe_validation_loss"] - trajectory["stable_continuation_validation_loss"]

print(f"Run: {LOG_DIR.name}")
print(f"metrics rows: {len(metrics)}")
print(f"sweep rows: {len(sweep)}")
print(f"trajectory rows: {len(trajectory)}")
print(f"starts: {sorted(sweep['probe_start_step'].unique())}")
print(f"ratios: {sorted(sweep['final_lr_ratio'].unique(), reverse=True)}")

## Stable Training Curve and Probe Starts

In [ ]:
train_rows = metrics[metrics["train_loss"].notna()].copy()
val_curve = metrics[metrics["validation_loss"].notna()].copy()
starts = sorted(sweep["probe_start_step"].unique())

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(val_curve["step"], val_curve["validation_loss"], color="tab:blue", linewidth=1.8, label="validation loss")
for step in starts:
    ax.axvline(step, color="tab:orange", alpha=0.35, linewidth=1)
ax.scatter(starts, val_rows[val_rows["probe_start_step"].isin(starts)]["probe_start_val_loss"], color="tab:orange", zorder=4, label="probe starts")
ax.set_xlabel("training step")
ax.set_ylabel("validation loss")
ax.set_title("Stable training validation loss with plateau probe starts")
ax.legend()
plt.tight_layout()
plt.show()

## Raw Final Validation Loss

This plot is useful descriptively, but do not use it alone to compare different start checkpoints. Later starts have more training.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for step, group in sweep.groupby("probe_start_step"):
    group = group.sort_values("final_lr_ratio", ascending=False)
    ax.plot(group["final_lr_ratio"], group["probe_final_val_loss"], marker="o", label=f"start {step}")
ax.set_xscale("log")
ax.invert_xaxis()
ax.set_xlabel("final LR ratio")
ax.set_ylabel("probe final validation loss")
ax.set_title("Raw final val loss after 150-step probes")
ax.legend(ncols=2)
plt.tight_layout()
plt.show()

## Decay Advantage vs Stable Continuation

This is the central analysis. Negative values mean the decay amount beats `final_lr_ratio = 1.0` from the same checkpoint over the same 150-step horizon.

In [ ]:
adv = sweep[sweep["final_lr_ratio"] != 1.0].copy()

fig, ax = plt.subplots(figsize=(10, 5))
for ratio, group in adv.groupby("final_lr_ratio"):
    group = group.sort_values("probe_start_step")
    ax.plot(group["probe_start_step"], group["decay_advantage"], marker="o", linewidth=1.8, label=f"{ratio:g}")
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("probe start step")
ax.set_ylabel("decay val - stable continuation val")
ax.set_title("Decay advantage by start checkpoint")
ax.legend(title="final LR ratio", ncols=2)
plt.tight_layout()
plt.show()

adv[["probe_start_step", "final_lr_ratio", "probe_final_val_loss", "stable_continuation_val_loss", "decay_advantage"]].sort_values(["probe_start_step", "decay_advantage"])

## Best Ratio Per Checkpoint

In [ ]:
best_by_start = sweep.loc[sweep.groupby("probe_start_step")["probe_final_val_loss"].idxmin()].copy()
best_by_start = best_by_start[[
    "probe_start_step", "probe_start_val_loss", "final_lr_ratio", "probe_final_val_loss",
    "stable_continuation_val_loss", "decay_advantage", "grad_snr", "grad_weight_ratio",
    "param_update_norm", "loss_improvement_rate"
]].sort_values("probe_start_step")
best_by_start

## Validation Trajectories During Probes

In [ ]:
for step, step_df in trajectory.groupby("probe_start_step"):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for ratio, group in step_df.groupby("final_lr_ratio"):
        group = group.sort_values("probe_decay_step")
        ax.plot(group["probe_decay_step"], group["probe_validation_loss"], marker="o", markersize=2.5, linewidth=1.2, label=f"{ratio:g}")
    ax.set_xlabel("probe decay step")
    ax.set_ylabel("validation loss")
    ax.set_title(f"Validation trajectories from start step {step}")
    ax.legend(title="ratio", ncols=3)
    plt.tight_layout()
    plt.show()

## Metric Links

These correlations ask whether metrics at the probe start predict decay advantage. Negative correlation means larger metric values are associated with more benefit from decay relative to stable continuation.

In [ ]:
metric_cols = [
    "probe_start_val_loss", "loss_variance", "loss_oscillation", "loss_improvement_rate",
    "grad_norm", "grad_snr", "grad_weight_ratio", "grad_cosine_sim",
    "adam_v_norm", "weight_norm", "param_update_norm",
]

rows = []
for ratio, group in adv.groupby("final_lr_ratio"):
    y = group["decay_advantage"].to_numpy()
    for metric in metric_cols:
        if metric not in group:
            continue
        x = group[metric].to_numpy()
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() < 3 or np.allclose(x[mask], x[mask][0]) or np.allclose(y[mask], y[mask][0]):
            continue
        pearson_r, pearson_p = stats.pearsonr(x[mask], y[mask])
        spearman_r, spearman_p = stats.spearmanr(x[mask], y[mask])
        rows.append({
            "final_lr_ratio": ratio,
            "metric": metric,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p,
        })
metric_corr = pd.DataFrame(rows).sort_values(["final_lr_ratio", "pearson_r"])
metric_corr

In [ ]:
if not metric_corr.empty:
    display(metric_corr.pivot(index="metric", columns="final_lr_ratio", values="pearson_r"))

## Interpretation Checklist

Look for:

- Does any ratio have negative decay advantage after step 1000?
- Does the best ratio shift from `1.0` or `0.5` toward `0.3/0.2/0.1` as the plateau develops?
- Do `grad_snr`, `grad_weight_ratio`, or `loss_improvement_rate` correlate with decay advantage?
- If no decay ratio beats `1.0`, the conclusion is that the model still benefits more from stable LR continuation than decay in this window.